## 1 概述

基础神经网络 MLP 有一个关键的结构性局限：它假设输入特征之间是相互独立的（不是线性无关的独立，而是顺序无关的独立）。在 MLP 的视角里，一个样本的第 1 个元素、第 2 个元素、... 、第 T 个元素，只是“特征1”、“特征2”、... 、“特征 T”，它们之间的先后顺序被完全忽略了。

而有一类问题，顺序就是信息，比如：无人机入侵检测（UAVIDS），攻击流量往往表现为“某些消息 ID 在一定时间段内大量重复出现”，因此需要一种能够利用顺序信息的模型。

RNN（循环神经网络，Recurrent Neural Network） 正是为此而设计。其核心思想是在时间维度上共享同一组权重，并通过一个“隐藏状态”将上一时刻的信息传递到当前时刻，从而让模型具备“记忆”能力。

## 2 序列问题

假设有三个样本，特征都是四个，其输入矩阵为
$$
X=\begin{pmatrix} 11 & 12 & 13 & 14 \\ 21 & 22 & 23 & 24 \\ 31 & 32 & 33 & 34 \end{pmatrix}
$$

假设是一个单隐藏层的 MLP 模型，$X$ 作为输入，在隐藏层通过一次矩阵运算即可得到输出。

若是一个单隐藏层的 RNN 模型，会将 $X$ 按时间（顺序）一列列分开：
$$
X_1=\begin{pmatrix} 11 \\ 21 \\ 31 \end{pmatrix} \quad,\quad
X_2=\begin{pmatrix} 12 \\ 22 \\ 32 \end{pmatrix} \quad,\quad
X_3=\begin{pmatrix} 13 \\ 23 \\ 33 \end{pmatrix}
$$

然后将 $X_t$ 循环依次输入隐藏层进行计算，每个循环得到的结果 $H$（意为隐藏状态，hidden state） 会参与到下个循环的计算中，即 $H_t$ 的计算需要用到 $H_{t-1}$ ，这样就让模型“记住了”顺序信息。

直到把整个输入矩阵 $X$ 中的列，循环处理结束，得到 $H_T$ ，本次隐藏层的计算才算完成。

> 样本之间本身就是独立的，在 MLP 中特征被视为并列的属性，没有先后顺序之分，因此输入矩阵 $X$ 的行和列都可以并行计算，一次矩阵运算即可完成一层的计算。  
> 但在 RNN 中，特征的位置被赋予了时间顺序的含义，那 $X$ 的列就只能串行计算了，因此按列拆分依次循环处理，循环结束才算完成一层的计算。

## 3 MLP vs RNN

**概念对比**

||输入层|隐藏层1|隐藏层2|输出层|
|-|-|-|-|-|
| MLP | $X$ | 一次矩阵运算得 $A_1$ | 一次矩阵运算得 $A_2$ | $\hat{Y}=\text{Sigmoid}(A_2W+b)$ |
| RNN | $X=[X_1,X_2,...,X_T]$ | 按列循环处理得 $[H_1^{[1]},H_2^{[1]},...,H_T^{[1]}]$ | 按列循环处理得 $[H_1^{[2]},H_2^{[2]},...,H_T^{[2]}]$ | $\hat{Y}=\text{Sigmoid}(H_T^{[2]}W+b)$ |

**公式对比**

||输入|线性部分|激活函数|备注|
|-|-|-|-|-|
| MLP | $X$ | $Z=XW+b$ | $A=\text{ReLU}(Z)$ | 一次矩阵运算即可完成 |
| RNN | $X_t,H_{t-1}$ | $Z_t=X_tW_{xh}+H_{t-1}W_{hh}+b_h$ | $H_t=\tanh(Z_t)$ | 循环处理完整个序列得到 $H_T$ 才算完成 <br> 循环开始前需要初始化 $H_0=0$|

**ReLU vs tanh**

||值域|在 MLP 中|在 RNN 中|备注|
|-|-|-|-|-|
| ReLU | $[0,+\infty)$ | 标准选择 | 无上限累加，数值容易爆炸 | |
| tanh | $(-1,1)$ | 可用，但不如 ReLU 效率 | 标准选择，可防止数值溢出 | |

**RNN 的权重共享**
- 在一层的计算中，MLP 只有一个权重，而 RNN 有两个 $W_{xh}\,,\,W_{hh}$ ：
    - $W_{xh}$ : 表示输入层 x 到隐藏层 h 的权重。用于处理当前的输入 $X_t$ 。
    - $W_{hh}$ : 表示隐藏层 h 到隐藏层 h 的权重。用于处理历史记忆 $H_{t-1}$ 。
    - 这两个权重在循环中保持不变（所以叫权重共享），多层RNN中每层都有自己独立的这俩权重，与 MLP 一样。
    - 与 MLP 一样，这两个权重是随机初始化的，在整个网络的前向传播中保持不变，在反向传播时更新。
- 在上文`概念对比表`中，RNN 的输出层准确的说应该是：$\hat{Y}=\text{Sigmoid}(H_T^{[2]}W_{hy}+b_{y})$ ，以表明 $W_{hy}$ 是隐藏层到输出层的权重，$b_y$ 是输出层的偏置。

**为什么把顺序称为时间维度**：按我理解，多层隐藏层的结构通常称为空间堆叠，因此层内用循环处理顺序信息就相应的称为时间堆叠。

- 空间维度的堆叠：每一层都有自己的权重矩阵 $W_{xh}^{[l]}\,,\,W_{hh}^{[l]}$ ，独立学习不同层次的特征。
- 时间维度的堆叠：在同一层内，所有时间步共享同一组权重 $W_{xh}^{[l]}\,,\,W_{hh}^{[l]}$ ，实现对顺序的记忆（位置不变性）。

## 4 反向传播

在 MLP 中，误差的反向传播只存在于空间上（层间），而 RNN 还要在时间上（层内循环间）传播，所以 RNN 的反向传播被称为 BPTT (Backpropagation Through Time) ，这种二维传播可以形象的称呼为“空间层”和“时间步”。

空间层的传播 RNN 与 MLP 一致，主要分析下 RNN 时间步的传播。

在 MLP 中我们已知其空间层的反向传播公式是：$\delta^{[l]}=(\delta^{[l+1]}\cdot (W^{[l+1]})^T)\odot 1_{z^{[l]}>0}$

> 注：为与表示时间步数的下标区分，这里用上标表示空间层数

时间步的传播只需要把层数上标改成时间下标，然后把 ReLU 导数改成 tanh 的导数即可：$\delta_t=(\delta_{t+1}\cdot W_{hh}^T)\odot \tanh^{'}(Z_t)$

已知 $\tanh^{'}(x)=1-\tanh^2(x)$ 且 $H_t=\tanh(Z_t)$ ，所以：
$$
\delta_t=(\delta_{t+1}\cdot W_{hh}^T)\odot (1-H_t^2)
$$

## 5 梯度消失

当误差在时间上传播时，从最后的时刻 T 传回第一个时刻 1 ，需要连续应用上述公式递推 T-1 次：
$$
\delta_{T-1}=(\delta_{T}\cdot W_{hh}^T)\odot (1-H_{T-1}^2) \\
\downarrow \\
\delta_{T-2}=(((\delta_{T}\cdot W_{hh}^T)\odot (1-H_{T-1}^2))\cdot W_{hh}^T)\odot (1-H_{T-2}^2) \\
\downarrow \\
\cdots \\
\downarrow \\
\delta_{1}=(...((\delta_{T}\cdot W_{hh}^T)\odot (1-H_{T-1}^2))\cdot W_{hh}^T\odot (1-H_{T-2}^2)...)\cdot W_{hh}^T\odot (1-H_{1}^2) \\
$$

- 每一次递推，误差就要先右乘 $W_{hh}^T$ ，再逐元素乘 $1-H_t^2$ 。
- 而我们知道，tanh 的值域是 $(-1,1)$ ，其导数 $1-H_t^2$ 的值域为 $(0,1]$ ，每乘一次都是衰减，当序列长度很长时，连乘的结果必然趋近于 0 。
- 而连乘 $W_{hh}^T$ ，若特征小于 1 ，会加速梯度的衰减；若特征大于 1 ，则会放大梯度可能导致梯度爆炸，再加上 $1-H_t^2$ 的压缩，训练容易不稳定。

还可以从链式法则的角度理解：
$$
\frac{\partial J}{\partial Z_1}=\frac{\partial J}{\partial Z_T}\cdot \frac{\partial Z_T}{\partial Z_{T-1}}\cdot \frac{\partial Z_{T-1}}{\partial Z_{T-2}}\cdots \frac{\partial Z_2}{\partial Z_1}
$$

每一项 $\frac{\partial Z_{t+1}}{\partial Z_{t}}$ 都在 0 到 1 之间，当 T 很大时，连乘结果趋近于 0 。

> 为解决这个问题，LSTM 算法引入了细胞状态 $C_t$ 使误差在时间上反向传播时不经过 $W_{hh}^T$ 和 $\tanh^{'}$ 的连乘，而是通过加法传递，从根本上避免了梯度衰减。
